# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# Import the necessary libs
import os
import json
import uuid

import chromadb

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

from lib.parsers import PydanticOutputParser

from pydantic import BaseModel, Field
from typing import List, Optional

from tavily import TavilyClient
from chromadb.utils import embedding_functions

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [4]:
# Environment variables are loaded in the setup cell above.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
# Instantiate your ChromaDB Client
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [6]:
# Pick one embedding function
# Part 2 rebuilds this same function when it loads the collection,
# so if you pick something different, keep it identical in both notebooks.
open_api_key = os.getenv("OPENAI_API_KEY")

if open_api_key.startswith("ollama"):
    # Local workspace
    embedding_fn = embedding_functions.OllamaEmbeddingFunction(
        model_name="nomic-embed-text",
        url="http://localhost:11434/api/embeddings"
    )
else:
    # OpenAI workspace
    # Udacity workspace
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key_env_var="OPENAI_API_KEY",
        api_base=os.getenv("OPENAI_BASE_URL"),
    )

In [7]:
# Create a collection
# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [8]:
# Create retrieve_game tool
# It should use the chroma client and collection you created in Part 1 (same path and name)
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Source: Source file name
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    - Similarity: Similarity score
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["metadatas", "documents", "distances"]
    )
    
    docs = []
    for i, (meta, dist) in enumerate(zip(
            results['metadatas'][0], 
            results['distances'][0]
        )):
        docs.append({
            "Source": meta.get("Source"),
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "Similarity": 1 - dist
        })

    return docs

#### Evaluate Retrieval Tool

In [9]:
from typing import Dict, List

class RetrievalEvaluationResult(BaseModel):
    """Metrics for task completion evaluation"""
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Description about the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> RetrievalEvaluationResult:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "You are a evaluator following strict rules."
        "Paraphrase the question to identity the key entity and the core information asked."
        "Evaluate whether the retrieved documents contain information that directly answers the question.\n"
        "You MUST consider similarity scores.\n"
        "You MUST ignore all previous conversation context.\n"
        "Mark useful=False if:\n"
        "- The documents do not mention the game asked about\n"
        "- The documents do not contain the specific information needed\n"
        "- The documents only contain unrelated metadata\n\n"
        f"Question:\n{question}\n\n"
        "Retrieved documents:\n"
        f"{json.dumps(retrieved_docs, indent=2)}\n\n"
        "Return a JSON object with:\n"
        "- useful: boolean\n"
        "- description: detailed explanation\n"
    )

    llm = LLM(model="gpt-4o")  # uses OPENAI_API_KEY / OPENAI_BASE_URL from env

    # Use structured output with Pydantic model
    response = llm.invoke(
        input=prompt, 
        response_format=RetrievalEvaluationResult
    )
    
    # Parse the structured response
    parser = PydanticOutputParser(model_class=RetrievalEvaluationResult)
    try:
        evaluation = parser.parse(response)

        return evaluation
    except Exception as e:
        print(f"Debug: Structured parsing error: {e}")
        
        return RetrievalEvaluationResult(useful=False, description=e)

#### Game Web Search Tool

In [10]:
# Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(question: str):
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    results = tavily_client.search(
        query=question,
        max_results=5,
        include_raw_content=False,
        search_depth='basic',
    )

    # Normalize to a simple list of {title, url, content}
    pages = []
    for r in results.get("results", []):
        pages.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content")[:200],
        })

    return pages

### Agent

In [16]:
# Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
from pydantic import BaseModel, Field
from typing import Optional

class AgentAnswer(BaseModel):
    answer: str = Field(description="The direct answer to the user's question.")
    source: str = Field(description="The source file name(s) of the information, either the internal source file name or web url(s).")
    source_type: str = Field(description="Either 'Internal', or 'Web Search'")
    fallback_used: bool = Field(description="True if a web search was required to answer the question.")
    internal_reasoning: str = Field(description="The agent's step-by-step reasoning for its conclusion.")
    
system_prompt = f"""
You are UdaPlay — a stateful AI Research Agent for the video game industry.

You MUST ALWAYS respond in the following JSON structure, matching the AgentAnswer Pydantic model:

{json.dumps(AgentAnswer.model_json_schema(), indent=2)}

Your responsibilities:
1. Use tools correctly and strategically
   - First attempt retrieval using the internal vector database via `retrieve_game`.
   - Second use `evaluate_retrieval` to judge whether retrieved documents are sufficient.
   - Only if retrieval is insufficient, call `game_web_search` to gather authoritative external information.
   - Never guess when a tool can provide factual information.

2. Reason step-by-step
   - Always populate `internal_reasoning` with your reasoning in natural language.
   - Keep `answer` focused on what the user needs to know.
   - Do NOT reveal tool internals, function signatures, or implementation details.
   - Do NOT mention the state machine or workflow.

3. Produce structured, reliable answers
   - When appropriate, structure the content inside `answer` using lists, tables, or JSON-like substructures.
   - When using web search, ALWAYS include citations (URLs) inside `answer`.
   - When answering from internal RAG, citations are optional and usually not required.

4. Be factual and precise
   - If uncertain, use tools instead of guessing.
   - If a question cannot be answered, state the limitation clearly in `answer` and explain why in `internal_reasoning`.

Your goal:
Provide authoritative, well-cited, and context-aware answers about video games, platforms, release dates, publishers, genres, and related industry knowledge, ALWAYS formatted as a valid AgentAnswer JSON object.
"""
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=system_prompt,
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)


In [17]:
# Demonstrate memory and session management
# The agent maintains conversation state - let's test this with follow-up questions

print("=== Demonstrating Stateful Memory ===")

# First question
first_query = "Tell me about Nintendo games in our database"
print(f"Query 1: {first_query}")

run1 = agent.invoke(first_query)
state1 = run1.get_final_state()
print(f"Answer 1: {state1['messages'][-1].content}")
print(f"Messages in session: {len(state1['messages'])}")

# Follow-up question that references the previous conversation
followup_query = "Which of those were released before 2000?"
print(f"\nQuery 2 (follow-up): {followup_query}")

run2 = agent.invoke(followup_query)
state2 = run2.get_final_state()
print(f"Answer 2: {state2['messages'][-1].content}")
print(f"Messages in session: {len(state2['messages'])}")

# Demonstrate session reset
print("\n=== Session Management ===")
print(f"Session runs before reset: {len(agent.get_session_runs())}")

# Start a new session
new_session_query = "What racing games do we have?"
run3 = agent.invoke(new_session_query, session_id="racing_session")
state3 = run3.get_final_state()
print(f"New session answer: {state3['messages'][-1].content}")

# Show that different sessions have different histories
print(f"Default session runs: {len(agent.get_session_runs('default'))}")
print(f"Racing session runs: {len(agent.get_session_runs('racing_session'))}")

=== Demonstrating Stateful Memory ===
Query 1: Tell me about Nintendo games in our database
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer 1: {
  "answer": {
    "games": [
      {
        "name": "Pokémon Gold and Silver",
        "platform": "Game Boy Color",
        "year_of_release": 1999,
        "description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics."
      },
      {
        "name": "Super Mario 64",
        "platform": "Nintendo 64",
        "year_of_release": 1996,
        "description": "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."
      },
      {
  

In [18]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
    "Who published Grand Theft Auto: San Andreas?",
    "What platform was Super Mario 64 launched on?",
    "What is Rockstar Games working on right now?"
]

def make_uuid():
    return str(uuid.uuid4())


def generate_agent_report(agent, queries=queries):
    print("# UdaPlay Agent Performance Report\n")

    for q in queries:
        print(f"## 🎮 Query: {q}\n")
        print("**Note:** Running in separate session")

        run = agent.invoke(q, session_id=make_uuid())
        final_state = run.get_final_state()
        messages = final_state["messages"]

        tool_blocks = []
        final_raw = messages[-1].content

        # Try to parse AgentAnswer JSON
        try:
            answer_obj = AgentAnswer.model_validate_json(final_raw)
        except Exception:
            answer_obj = None

        for msg in messages:
            if isinstance(msg, ToolMessage):
                tool_blocks.append(f"- **{msg.name}** → {msg.content}")

        print("\n### 🛠️ Tools Used")
        if tool_blocks:
            for t in tool_blocks:
                print(t)
        else:
            print("- No tools used")

        print("\n### ✅ Final Answer")
        print(final_raw if answer_obj is None else answer_obj.answer)

        if answer_obj is not None:
            print("\n### 🧠 Reasoning")
            print(answer_obj.internal_reasoning)

            print("\n### 🔗 Source")
            print(f"Type: {answer_obj.source_type}")
            print(f"Source: {answer_obj.source}")

        print("\n" + "-"*80 + "\n")

In [19]:
generate_agent_report(agent)

# UdaPlay Agent Performance Report

## 🎮 Query: When was Pokémon Gold and Silver released?

**Note:** Running in separate session
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

### 🛠️ Tools Used
- **retrieve_game** → "[{'Source': '006.json', 'Platform': 'Game Boy Color', 'Name': 'Pok\u00e9mon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.', 'Similarity': 0.7453724145889282}, {'Source': '007.json', 'Platform': 'Game Boy Advance', 'Name': 'Pok\u00e9mon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pok\u00e9mon games set in the Hoenn region, featuring new

### (Optional) Advanced — Stand Out

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes